<a href="https://colab.research.google.com/github/kaiju-no-9/Pytorch-Notes-/blob/main/class_7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 7b: Why RAG Breaks & What Comes Next
## Advanced RAG + PageIndex (Vectorless, Reasoning-based RAG)

By the end of this notebook, you will have:
1. **Seen context rot in action** - watched a model degrade as you add more context
2. **Built advanced RAG** - hybrid search, reranking, query transformation (HyDE)
3. **Used PageIndex** - vectorless, reasoning-based RAG on a real PDF
4. **Compared all approaches** on document understanding

---

## Part 0: Setup

In [ ]:
!pip install -q openai numpy rank_bm25 sentence-transformers

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
print("[OK] API key loaded")

[OK] API key loaded


In [ ]:
from openai import OpenAI
import json, numpy as np, time, re

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

MODEL = "qwen/qwen3.5-122b-a10b"

def chat(messages, model=MODEL, temperature=0.3, max_tokens=1024):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

print(chat([{"role": "user", "content": "Say 'hello world' and nothing else."}]))

hello world


---
## Part 1: See Context Rot In Action

We use a **multi-hop** reasoning task (not just needle-in-haystack) because Chroma's research shows context rot is far worse when the model needs to connect information across the context.

The model must find TWO separate facts and combine them to answer correctly.

In [ ]:
# === MULTI-HOP NEEDLE ===
# The model must find TWO facts and connect them:
# Fact 1: "Dr. Elena Vasquez leads the AURORA-7 initiative."
# Fact 2: "Dr. Vasquez confirmed the launch window opens on March 15, 2024."
# Question: "What is the launch date for AURORA-7?"
# The model needs to connect Vasquez -> AURORA-7 -> March 15.

FACT_A = "Dr. Elena Vasquez was appointed as the lead researcher for the classified AURORA-7 initiative after her successful tenure directing the Meridian program."
FACT_B = "In a private briefing last Tuesday, Dr. Vasquez confirmed that the launch window for her current project opens on March 15, 2024, pending final safety reviews."

# Filler paragraphs about similar topics (projects, dates, people)
FILLER_POOL = [
    "The quarterly infrastructure review highlighted upgrades to the east-coast data centers. Migration timelines were set for late Q2. The operations team reported a 99.97 percent uptime for the previous period and proposed additional redundancy measures for the backup systems.",
    "Dr. James Morton presented findings from the computational biology lab. The new protein folding algorithm showed a 34 percent improvement in prediction accuracy. Funding proposals were submitted to three federal agencies for continued research.",
    "Project TITAN-6 entered Phase 3 clinical evaluation under the supervision of Dr. Sarah Lin. Preliminary results are expected by August 2024. The regulatory affairs team has begun preparing submission documents for the oversight board.",
    "The advanced materials team tested a new ceramic composite for thermal shielding. Results exceeded expectations at temperatures above 2000 degrees Celsius. A patent application was filed and manufacturing partners have been contacted.",
    "Chief Financial Officer Mark Reynolds presented the revised budget allocations for fiscal year 2025. Research and development spending will increase by 18 percent. Capital expenditure for new laboratory facilities was approved at 47 million dollars.",
    "The cybersecurity division completed its annual red team exercise across all classified networks. Two medium-severity vulnerabilities were discovered and patched within 72 hours. New endpoint detection protocols were deployed organization-wide.",
    "Professor Lisa Chang published her team's findings on quantum entanglement stability in the latest issue of Physical Review Letters. The results suggest a viable path toward error-corrected quantum computing within the next decade.",
    "Project ORION-11 received continued funding approval from the defense advisory panel. The next milestone review is scheduled for May 2024. Program manager David Kowalski noted that the project remains on schedule and within budget parameters.",
    "The talent acquisition team reported hiring 43 new researchers across six departments. Retention rates improved to 94 percent following the introduction of flexible work arrangements. The mentorship program expanded to include 120 pairs.",
    "Environmental monitoring stations detected a slight increase in seismic activity near the northern test facility. Geological surveys confirmed no structural risk. Additional monitoring equipment has been installed as a precautionary measure.",
    "The satellite communications upgrade was completed two weeks ahead of schedule. Bandwidth capacity doubled to support the growing number of field operations. Dr. Robert Kim oversaw the final integration testing phase and signed off on deployment.",
    "Project NEBULA-4 concluded its preliminary design review. The propulsion subsystem exceeded thrust-to-weight requirements by 12 percent. Systems engineering lead Patricia Gomez recommended advancing to the detailed design phase in Q3.",
    "The medical research wing reported promising results from the Phase 2 trial of compound MRX-7821. Efficacy rates reached 67 percent in the target population. An expanded trial involving 2000 additional participants was approved for the next quarter.",
    "Facilities management completed the renovation of Building 7, adding 15000 square feet of laboratory space. The new cleanroom meets ISO Class 5 standards. Occupancy is planned for early February 2024.",
    "Dr. Amanda Foster's team demonstrated a novel machine learning approach for anomaly detection in network traffic. The system reduced false positive rates by 40 percent compared to existing methods used across the organization.",
    "The supply chain optimization initiative yielded a 22 percent reduction in procurement lead times. Strategic partnerships with three new vendors were established. Inventory carrying costs decreased by 1.3 million dollars annually.",
    "The HELIOS-8 solar energy research program completed its first full year of field testing. Energy conversion efficiency averaged 31.4 percent across all test sites. Results will be presented at the International Energy Conference in Geneva.",
    "Human resources launched the updated professional development framework. Over 300 employees enrolled in the advanced leadership track. The tuition reimbursement budget was increased by 25 percent to accommodate growing demand.",
    "The autonomous systems laboratory tested its latest navigation algorithm in simulated urban environments. The system achieved a 98.7 percent obstacle avoidance rate. Real-world field trials are planned for the second half of 2024.",
    "Project AEGIS-3 was formally closed after achieving all deliverables ahead of schedule. The final report documented 14 technical innovations and 6 patent filings. Lessons learned sessions were conducted with all participating teams."
]

print(f"Fact A: {FACT_A}")
print(f"Fact B: {FACT_B}")
print(f"Filler paragraphs available: {len(FILLER_POOL)}")

Fact A: Dr. Elena Vasquez was appointed as the lead researcher for the classified AURORA-7 initiative after her successful tenure directing the Meridian program.
Fact B: In a private briefing last Tuesday, Dr. Vasquez confirmed that the launch window for her current project opens on March 15, 2024, pending final safety reviews.
Filler paragraphs available: 20


In [ ]:
# === CONTEXT ROT EXPERIMENT ===

QUESTION = "What is the launch date for AURORA-7?"
# Correct answer requires: Fact A (Vasquez leads AURORA-7) + Fact B (Vasquez launch = March 15)

def build_multihop_haystack(n_filler):
    """Build a haystack with TWO facts separated by filler."""
    fillers = (FILLER_POOL * ((n_filler // len(FILLER_POOL)) + 1))[:n_filler]
    np.random.seed(42)
    np.random.shuffle(fillers)
    # Place facts far apart - Fact A near start, Fact B near end
    pos_a = max(1, len(fillers) // 4)
    pos_b = (3 * len(fillers)) // 4
    fillers.insert(pos_a, FACT_A)
    fillers.insert(pos_b, FACT_B)
    return "\n\n".join(fillers)

def test_multihop(context, question):
    msg = [
        {"role": "system", "content": "Answer the question based ONLY on the provided context. Be precise and specific."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    return chat(msg, max_tokens=200)

experiments = [
    {"name": "Tiny (2 facts + 5 fillers)",    "n_filler": 5},
    {"name": "Small (2 facts + 20 fillers)",   "n_filler": 20},
    {"name": "Medium (2 facts + 60 fillers)",  "n_filler": 60},
    {"name": "Large (2 facts + 120 fillers)",  "n_filler": 120},
    {"name": "XL (2 facts + 200 fillers)",     "n_filler": 200},
]

print("CONTEXT ROT EXPERIMENT (Multi-Hop)")
print("=" * 60)
print(f"Question: {QUESTION}")
print(f"Correct: March 15, 2024 (requires connecting Vasquez -> AURORA-7 -> date)")
print()

for exp in experiments:
    ctx = build_multihop_haystack(exp["n_filler"])
    n_words = len(ctx.split())
    approx_tokens = n_words * 4 // 3
    print(f"\n>> {exp['name']} (~{n_words} words, ~{approx_tokens} tokens)")
    answer = test_multihop(ctx, QUESTION)
    correct = "march 15" in answer.lower()
    status = "[PASS]" if correct else "[FAIL]"
    print(f"   {status} Answer: {answer[:250]}")
    time.sleep(1)

print("\n" + "=" * 60)
print("Multi-hop reasoning degrades much faster than simple lookup.")
print("The model must connect Vasquez -> AURORA-7 -> March 15 date.")
print("As context grows, the attention budget gets stretched thin.")
print("This is context rot.")

CONTEXT ROT EXPERIMENT (Multi-Hop)
Question: What is the launch date for AURORA-7?
Correct: March 15, 2024 (requires connecting Vasquez -> AURORA-7 -> date)


>> Tiny (2 facts + 5 fillers) (~222 words, ~296 tokens)
   [PASS] Answer: Based on the provided context, the launch window for Dr. Vasquez's current project (AURORA-7) opens on **March 15, 2024**, pending final safety reviews.

>> Small (2 facts + 20 fillers) (~711 words, ~948 tokens)
   [PASS] Answer: Based on the provided context, the launch window for the AURORA-7 initiative opens on **March 15, 2024**, pending final safety reviews. This information was confirmed by Dr. Vasquez, who is the lead researcher for the project.

>> Medium (2 facts + 60 fillers) (~2037 words, ~2716 tokens)
   [PASS] Answer: Based on the provided context, Dr. Elena Vasquez is the lead researcher for the classified AURORA-7 initiative, and she confirmed that the launch window for her current project opens on **March 15, 2024**, pending final safety rev

## Part 2: Advanced RAG - Making Retrieval Smarter

In [ ]:
DOCUMENTS = [
    {"id": 1, "title": "Authentication System Overview",
     "content": "The authentication system uses OAuth 2.0 with PKCE flow for all client applications. Tokens are issued with a 1-hour expiry and refresh tokens last 30 days. Multi-factor authentication is required for admin accounts. The auth service is deployed on Kubernetes with 3 replicas for high availability. Rate limiting is set to 100 requests per minute per IP."},
    {"id": 2, "title": "API Rate Limiting Policy",
     "content": "All API endpoints enforce rate limiting based on the client subscription tier. Free tier: 60 requests/minute. Pro tier: 600 requests/minute. Enterprise: 6000 requests/minute. Rate limit headers (X-RateLimit-Remaining, X-RateLimit-Reset) are included in every response. When limits are exceeded, the API returns HTTP 429 with a Retry-After header."},
    {"id": 3, "title": "Error Handling Guide",
     "content": "HTTP 400: Bad request. HTTP 401: Unauthorized, token may be expired, refresh it. HTTP 403: Forbidden, insufficient permissions. HTTP 404: Not found. HTTP 429: Rate limited, wait and retry with exponential backoff. HTTP 500: Server error, retry after 30 seconds, contact support if persistent."},
    {"id": 4, "title": "Deployment Runbook",
     "content": "Production deployments happen daily at 2 AM UTC via automated CI/CD pipeline. Canary deployments are used with 5 percent traffic for 30 minutes before full rollout. Rollback procedure: run kubectl rollout undo in the production namespace. Health checks must pass for 5 consecutive minutes before traffic is shifted."},
    {"id": 5, "title": "Database Schema: Users Table",
     "content": "The users table contains: id (UUID primary key), email (varchar unique indexed), password_hash (varchar), created_at (timestamp), last_login (timestamp), subscription_tier (enum: free, pro, enterprise), mfa_enabled (boolean default false), api_key_hash (varchar nullable). Soft deletes are used via a deleted_at column."},
    {"id": 6, "title": "Incident Response: Auth Service Outage Jan 2024",
     "content": "On January 12 2024 the authentication service experienced a 47-minute outage due to database connection pool exhaustion. Root cause: a deployment introduced a query that held connections for 30+ seconds under load. Impact: approximately 12000 users unable to log in. Resolution: deployment rolled back and query optimized."},
    {"id": 7, "title": "API Versioning Strategy",
     "content": "The API uses URL-based versioning (e.g. /v1/ and /v2/). Breaking changes require a new major version. Non-breaking additions are added to the current version. Deprecated endpoints return a Sunset header with the removal date. Version v1 will be sunset on June 30 2025. All clients must migrate to v2 by that date."},
    {"id": 8, "title": "Monitoring and Alerting Setup",
     "content": "We use Prometheus for metrics collection and Grafana for dashboards. Critical alerts: p99 latency above 500ms, error rate above 1 percent, pod restarts above 3 in 5 minutes. Warning alerts: p95 latency above 200ms, CPU usage above 80 percent sustained for 10 minutes. Alerts route to PagerDuty for critical and Slack for warnings."},
    {"id": 9, "title": "Security: Token Refresh Flow",
     "content": "When an access token expires the client sends the refresh token to POST /auth/refresh. The server validates the refresh token, checks it has not been revoked, and issues a new access token plus new refresh token (rotation). The old refresh token is immediately invalidated. If a revoked refresh token is used ALL tokens for that user are invalidated."},
    {"id": 10, "title": "Onboarding: New Developer Setup",
     "content": "Step 1: Clone the monorepo from GitHub. Step 2: Run make setup to install dependencies. Step 3: Copy .env.example to .env and fill in local database credentials. Step 4: Run make migrate for database schema. Step 5: Run make seed for test data. Step 6: Run make dev to start the server on localhost:3000."}
]

print("Loaded %d documents" % len(DOCUMENTS))
for d in DOCUMENTS:
    print("   Doc %d: %s" % (d['id'], d['title']))

Loaded 10 documents
   Doc 1: Authentication System Overview
   Doc 2: API Rate Limiting Policy
   Doc 3: Error Handling Guide
   Doc 4: Deployment Runbook
   Doc 5: Database Schema: Users Table
   Doc 6: Incident Response: Auth Service Outage Jan 2024
   Doc 7: API Versioning Strategy
   Doc 8: Monitoring and Alerting Setup
   Doc 9: Security: Token Refresh Flow
   Doc 10: Onboarding: New Developer Setup


### 2.1 Semantic Search (recap from 7a)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_texts = ["%s. %s" % (d['title'], d['content']) for d in DOCUMENTS]
doc_embeddings = embedder.encode(doc_texts, normalize_embeddings=True)

def semantic_search(query, top_k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores = np.dot(doc_embeddings, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(DOCUMENTS[i], float(scores[i])) for i in top_idx]

query = "How do I handle an expired authentication token?"
results = semantic_search(query)
print("Query: %s\n" % query)
for doc, score in results:
    print("   [%.3f] Doc %d: %s" % (score, doc['id'], doc['title']))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: How do I handle an expired authentication token?

   [0.573] Doc 9: Security: Token Refresh Flow
   [0.409] Doc 1: Authentication System Overview
   [0.351] Doc 3: Error Handling Guide


### 2.2 BM25 Search (keyword-based)

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_docs = [text.lower().split() for text in doc_texts]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(DOCUMENTS[i], float(scores[i])) for i in top_idx]

query = "HTTP 429 rate limit error"
print("BM25 Query: %s\n" % query)
for doc, score in bm25_search(query):
    print("   [%.3f] Doc %d: %s" % (score, doc['id'], doc['title']))

print("\n--- Compare with semantic search ---\n")
for doc, score in semantic_search(query):
    print("   [%.3f] Doc %d: %s" % (score, doc['id'], doc['title']))

print("\n--> BM25 catches exact keyword matches that semantic search might rank differently.")

BM25 Query: HTTP 429 rate limit error

   [5.688] Doc 2: API Rate Limiting Policy
   [4.177] Doc 3: Error Handling Guide
   [1.546] Doc 8: Monitoring and Alerting Setup

--- Compare with semantic search ---

   [0.620] Doc 2: API Rate Limiting Policy
   [0.615] Doc 3: Error Handling Guide
   [0.309] Doc 6: Incident Response: Auth Service Outage Jan 2024

--> BM25 catches exact keyword matches that semantic search might rank differently.


### 2.3 Hybrid Search (BM25 + Semantic via RRF)

In [ ]:
def hybrid_search(query, top_k=3, k=60):
    """Reciprocal Rank Fusion: combines rankings from BM25 and semantic search."""
    sem_results = semantic_search(query, top_k=len(DOCUMENTS))
    bm25_results = bm25_search(query, top_k=len(DOCUMENTS))
    sem_ranks = {doc['id']: rank for rank, (doc, _) in enumerate(sem_results)}
    bm25_ranks = {doc['id']: rank for rank, (doc, _) in enumerate(bm25_results)}
    rrf_scores = {}
    for doc in DOCUMENTS:
        did = doc['id']
        rrf_scores[did] = 1.0/(k + sem_ranks[did]) + 1.0/(k + bm25_ranks[did])
    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [(next(d for d in DOCUMENTS if d['id']==did), rrf_scores[did]) for did in sorted_ids]

query = "What happens when I get a 429 error on the authentication endpoint?"
print("Query: %s\n" % query)
print("Semantic only:")
for doc, _ in semantic_search(query): print("   Doc %d: %s" % (doc['id'], doc['title']))
print("\nBM25 only:")
for doc, _ in bm25_search(query): print("   Doc %d: %s" % (doc['id'], doc['title']))
print("\nHybrid (RRF):")
for doc, score in hybrid_search(query): print("   [%.4f] Doc %d: %s" % (score, doc['id'], doc['title']))
print("\n--> Hybrid captures BOTH semantic meaning AND keyword matches.")

Query: What happens when I get a 429 error on the authentication endpoint?

Semantic only:
   Doc 3: Error Handling Guide
   Doc 6: Incident Response: Auth Service Outage Jan 2024
   Doc 1: Authentication System Overview

BM25 only:
   Doc 2: API Rate Limiting Policy
   Doc 1: Authentication System Overview
   Doc 9: Security: Token Refresh Flow

Hybrid (RRF):
   [0.0325] Doc 1: Authentication System Overview
   [0.0323] Doc 2: API Rate Limiting Policy
   [0.0323] Doc 6: Incident Response: Auth Service Outage Jan 2024

--> Hybrid captures BOTH semantic meaning AND keyword matches.


### 2.4 Reranking with LLM

In [ ]:
def rerank(query, candidates, top_k=3):
    candidate_text = ""
    for i, (doc, _) in enumerate(candidates):
        candidate_text += "\n[%d] %s: %s..." % (i, doc['title'], doc['content'][:200])
    prompt = 'Given the query: "%s"\n\nRank these documents by relevance. Return ONLY a JSON array of indices.\nExample: [2, 0, 4, 1, 3]\n\nDocuments:%s\n\nRanking (JSON array only):' % (query, candidate_text)
    msg = [{"role": "user", "content": prompt}]
    resp = chat(msg, max_tokens=100)
    try:
        match = re.search(r'\[.*?\]', resp)
        if match:
            ranking = json.loads(match.group())
            return [(candidates[i][0], len(candidates)-rank) for rank, i in enumerate(ranking) if i < len(candidates)][:top_k]
    except (json.JSONDecodeError, IndexError, TypeError):
        pass
    return [(doc, score) for doc, score in candidates[:top_k]]

query = "How do I debug a login failure related to token expiry or rate limiting?"
print("Query: %s\n" % query)
candidates = hybrid_search(query, top_k=6)
print("Before reranking (hybrid top-6):")
for doc, _ in candidates: print("   Doc %d: %s" % (doc['id'], doc['title']))
reranked = rerank(query, candidates, top_k=3)
print("\nAfter LLM reranking (top-3):")
for doc, _ in reranked: print("   Doc %d: %s" % (doc['id'], doc['title']))

Query: How do I debug a login failure related to token expiry or rate limiting?

Before reranking (hybrid top-6):
   Doc 9: Security: Token Refresh Flow
   Doc 6: Incident Response: Auth Service Outage Jan 2024
   Doc 3: Error Handling Guide
   Doc 1: Authentication System Overview
   Doc 2: API Rate Limiting Policy
   Doc 7: API Versioning Strategy

After LLM reranking (top-3):
   Doc 3: Error Handling Guide
   Doc 9: Security: Token Refresh Flow
   Doc 2: API Rate Limiting Policy


### 2.5 HyDE - Hypothetical Document Embeddings

Instead of searching with the *question*, generate a *hypothetical answer* and search with that.

In [ ]:
def hyde_search(query, top_k=3):
    msg = [{"role": "user", "content": "Write a short, factual paragraph that would answer this question. Do not say you don't know.\n\nQuestion: %s\n\nHypothetical answer:" % query}]
    hypo_answer = chat(msg, max_tokens=200)
    print("   Hypothetical answer: %s..." % hypo_answer[:150])
    hypo_emb = embedder.encode([hypo_answer], normalize_embeddings=True)
    scores = np.dot(doc_embeddings, hypo_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(DOCUMENTS[i], float(scores[i])) for i in top_idx]

query = "Our users can't log in, what could be wrong?"
print("Query: %s\n" % query)
print("Standard semantic search:")
for doc, score in semantic_search(query): print("   [%.3f] Doc %d: %s" % (score, doc['id'], doc['title']))
print("\nHyDE search:")
for doc, score in hyde_search(query): print("   [%.3f] Doc %d: %s" % (score, doc['id'], doc['title']))
print("\n--> HyDE generates a hypothetical answer mentioning tokens, auth, etc.")
print("   This pulls in more relevant docs than the vague original query.")

Query: Our users can't log in, what could be wrong?

Standard semantic search:
   [0.471] Doc 6: Incident Response: Auth Service Outage Jan 2024
   [0.291] Doc 3: Error Handling Guide
   [0.178] Doc 5: Database Schema: Users Table

HyDE search:
   Hypothetical answer: Several technical issues could prevent users from logging in, including server-side outages, database connectivity failures, or authentication service...
   [0.644] Doc 6: Incident Response: Auth Service Outage Jan 2024
   [0.290] Doc 9: Security: Token Refresh Flow
   [0.284] Doc 3: Error Handling Guide

--> HyDE generates a hypothetical answer mentioning tokens, auth, etc.
   This pulls in more relevant docs than the vague original query.


### 2.6 Full Advanced RAG Pipeline

In [ ]:
def advanced_rag(query, use_hyde=True, use_rerank=True):
    print("\n" + "=" * 60)
    print("Query: %s" % query)
    print("   HyDE: %s | Reranking: %s" % ('ON' if use_hyde else 'OFF', 'ON' if use_rerank else 'OFF'))
    print("=" * 60)
    if use_hyde:
        print("\nStep 1: HyDE query transformation...")
        candidates = hyde_search(query, top_k=5)
    else:
        print("\nStep 1: Hybrid search...")
        candidates = hybrid_search(query, top_k=5)
    print("   Retrieved %d candidates" % len(candidates))
    if use_rerank:
        print("\nStep 2: Reranking...")
        final_docs = rerank(query, candidates, top_k=3)
    else:
        final_docs = candidates[:3]
    print("   Final docs:")
    for doc, _ in final_docs: print("      - Doc %d: %s" % (doc['id'], doc['title']))
    print("\nStep 3: Generating answer...")
    context = "\n\n".join("[%s]: %s" % (doc['title'], doc['content']) for doc, _ in final_docs)
    msg = [
        {"role": "system", "content": "Answer based on the provided context. Cite which document(s) you used."},
        {"role": "user", "content": "Context:\n%s\n\nQuestion: %s" % (context, query)}
    ]
    answer = chat(msg, max_tokens=500)
    print("\nAnswer: %s" % answer)
    return answer

advanced_rag("Our users are getting 429 errors when trying to log in during peak hours. How should we fix this?")


Query: Our users are getting 429 errors when trying to log in during peak hours. How should we fix this?
   HyDE: ON | Reranking: ON

Step 1: HyDE query transformation...
   Hypothetical answer: HTTP 429 errors indicate that your rate limiting policies are triggering because the volume of login requests exceeds the configured thresholds during...
   Retrieved 5 candidates

Step 2: Reranking...
   Final docs:
      - Doc 3: Error Handling Guide
      - Doc 2: API Rate Limiting Policy
      - Doc 1: Authentication System Overview

Step 3: Generating answer...

Answer: Based on the provided context, the 429 errors are caused by the rate limiting configuration on the authentication service. To address this, you should implement the recommended error handling strategy:

1.  **Identify the Cause:** According to the **[Authentication System Overview]**, the authentication service enforces a rate limit of **100 requests per minute per IP**. During peak hours, users exceeding this threshold wi

'Based on the provided context, the 429 errors are caused by the rate limiting configuration on the authentication service. To address this, you should implement the recommended error handling strategy:\n\n1.  **Identify the Cause:** According to the **[Authentication System Overview]**, the authentication service enforces a rate limit of **100 requests per minute per IP**. During peak hours, users exceeding this threshold will receive 429 errors.\n2.  **Implement Retry Logic:** The **[Error Handling Guide]** specifies that for HTTP 429 errors, clients should **"wait and retry with exponential backoff."**\n3.  **Utilize Headers:** The **[API Rate Limiting Policy]** states that when limits are exceeded, the API returns a **Retry-After header**, which should be used to determine the wait time before retrying.\n\n**Documents Used:**\n*   [Error Handling Guide]\n*   [API Rate Limiting Policy]\n*   [Authentication System Overview]'

---
## Part 3: PageIndex - Vectorless, Reasoning-based RAG

Instead of vectors, chunks, and embeddings, PageIndex builds a tree from the document and the LLM *reasons* over that tree to find relevant sections.

**Setup:** Students need a PageIndex API key from https://docs.pageindex.ai (free tier).

In [ ]:
!pip install -q pageindex

In [ ]:
from pageindex import PageIndexClient
import pageindex.utils as utils

PAGEINDEX_API_KEY = userdata.get("PAGEINDEX_API_KEY")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
print("[OK] PageIndex client initialized")

[OK] PageIndex client initialized


### 3.1 Submit a PDF and Build the Tree Index

In [ ]:
import requests as req

PDF_URL = "https://arxiv.org/pdf/2501.12948.pdf"
PDF_PATH = "./sample_paper.pdf"

print("Downloading sample PDF...")
resp = req.get(PDF_URL)
with open(PDF_PATH, "wb") as f:
    f.write(resp.content)
print("[OK] Downloaded: %d KB" % (len(resp.content) // 1024))

[OK] Downloaded: 4916 KB


In [ ]:
print("Submitting PDF to PageIndex for tree generation...")
print("   (This may take 1-3 minutes)")

result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]
print("   Document ID: %s" % doc_id)

while True:
    status = pi_client.get_document(doc_id)["status"]
    if status == "completed":
        print("[OK] Tree generation complete!")
        break
    elif status == "failed":
        print("[ERROR] Processing failed")
        break
    print("   Status: %s..." % status)
    time.sleep(10)

Submitting PDF to PageIndex for tree generation...
   (This may take 1-3 minutes)
   Document ID: pi-cmn394l1u05lfwopkn4d6y12t
   Status: processing...
   Status: processing...
   Status: processing...
   Status: processing...
   Status: processing...
   Status: processing...
[OK] Tree generation complete!


In [ ]:
tree = pi_client.get_tree(doc_id)

print("Document Tree Structure:")
print("=" * 60)

def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("|-- " if indent > 0 else "")
        page_range = "(pages %s-%s)" % (node.get('start_index', '?'), node.get('end_index', '?'))
        print("%s%s %s" % (prefix, node['title'], page_range))
        if node.get('summary'):
            s = node['summary']
            summary = s[:100] + '...' if len(s) > 100 else s
            print("%s  Summary: %s" % ('  ' * indent, summary))
        if node.get('nodes'):
            print_tree(node['nodes'], indent + 1)

if isinstance(tree, list):
    print_tree(tree)
elif isinstance(tree, dict) and 'nodes' in tree:
    print("Document: %s" % tree.get('doc_name', 'Unknown'))
    print_tree(tree['nodes'])
else:
    print(json.dumps(tree, indent=2)[:3000])

print("\n--> No chunks! No embeddings! Just the document's natural structure.")

Document Tree Structure:
{
  "doc_id": "pi-cmn394l1u05lfwopkn4d6y12t",
  "status": "completed",
  "retrieval_ready": true,
  "result": [
    {
      "title": "Abstract",
      "node_id": "0000",
      "page_index": 1,
      "text": "# Abstract\n\nGeneral reasoning represents a long-standing and formidable challenge in artificial intelligence. Recent breakthroughs, exemplified by large language models (LLMs) *(Brown et al., 2020; OpenAI, 2023)* and chain-of-thought prompting *(Wei et al., 2022b)*, have achieved considerable success on foundational reasoning tasks. However, this success is heavily contingent upon extensive human-annotated demonstrations, and models\u2019 capabilities are still insufficient for more complex problems. Here we show that the reasoning abilities of LLMs can be incentivized through pure reinforcement learning (RL), obviating the need for human-labeled reasoning trajectories. The proposed RL framework facilitates the emergent development of advanced reasoning p

### 3.2 Query with Reasoning-based Retrieval

In [ ]:
questions = [
    "What training methodology did they use for the reasoning model?",
    "What benchmarks did they evaluate on and what were the key results?",
    "How does their approach compare to OpenAI's o1?"
]

print("Querying with PageIndex (reasoning-based retrieval)")
print("=" * 60)

for q in questions:
    print("\nQuestion: %s" % q)
    print("-" * 40)
    response = pi_client.chat_completions(
        messages=[{"role": "user", "content": q}],
        doc_id=doc_id
    )
    answer = response["choices"][0]["message"]["content"]
    print("Answer: %s" % answer[:500])
    if len(answer) > 500:
        print("   ... (%d chars total)" % len(answer))
    time.sleep(2)

print("\n--> PageIndex REASONED about which sections to look at.")
print("   No vector similarity. Just tree navigation + reasoning.")

Querying with PageIndex (reasoning-based retrieval)

Question: What training methodology did they use for the reasoning model?
----------------------------------------
Answer: The paper describes **two training methodologies** for their reasoning models:

## **DeepSeek-R1-Zero** (Pure RL Approach)
- **Algorithm**: Group Relative Policy Optimization (**GRPO**) - a more efficient alternative to PPO
- **Key innovation**: **No supervised fine-tuning** before RL - trained directly on the base model (DeepSeek-V3-Base)
- **Rewards**: Rule-based only
  - **Accuracy rewards**: Correctness of final answers (verified against test cases for code, exact matches for math)
  - **For
   ... (1652 chars total)

Question: What benchmarks did they evaluate on and what were the key results?
----------------------------------------
Answer: I'll help you find the benchmarks and key results from the DeepSeek-R1 paper. Let me start by examining the document structure to locate the evaluation sections.Based on

### 3.3 Peek Behind the Curtain

Let's manually show the tree to our free LLM and watch it reason about where to look.

In [ ]:
def tree_to_text(nodes, indent=0):
    lines = []
    for node in nodes:
        prefix = "  " * indent
        lines.append("%s[%s] %s (pages %s-%s)" % (prefix, node.get('node_id', '?'), node['title'], node.get('start_index', '?'), node.get('end_index', '?')))
        if node.get('summary'):
            lines.append("%s  Summary: %s" % (prefix, node['summary'][:150]))
        if node.get('nodes'):
            lines.extend(tree_to_text(node['nodes'], indent + 1))
    return lines

if isinstance(tree, list):
    tree_text = "\n".join(tree_to_text(tree))
elif isinstance(tree, dict) and 'nodes' in tree:
    tree_text = "\n".join(tree_to_text(tree['nodes']))
else:
    tree_text = json.dumps(tree, indent=2)[:3000]

query = "What reward signals did they use during RL training?"
prompt = "You are navigating a research paper to answer a question.\n\nHere is the document's tree structure:\n\n%s\n\nQuestion: %s\n\nWhich sections (node IDs) would you look at? Explain your reasoning step by step." % (tree_text, query)

print("Question: %s" % query)
print("\nLLM reasoning over the tree:")
print("-" * 40)
print(chat([{"role": "user", "content": prompt}], max_tokens=500))

print("\n--> This is EXACTLY what PageIndex does under the hood.")
print("   The LLM reads the tree and REASONS about where to look.")
print("   Only those specific pages get sent for answer generation.")

Question: What reward signals did they use during RL training?

LLM reasoning over the tree:
----------------------------------------
To find information about reward signals used during RL training, I would look at the following sections:

## Recommended Sections to Examine

**Primary sections to look for:**
1. **Method section** (likely node_id "0002" or similar) - This would contain the technical details of the RL framework, including reward function design
2. **Experiments/Training Details section** - Would describe how rewards were implemented and measured
3. **Any section specifically titled "Reward Design," "Reward Signals," or "RL Framework"**

## Reasoning Step by Step

**Step 1: Analyze what's available**
- The provided tree shows only "Abstract" (0000) and "1 Introduction" (0001)
- These sections provide high-level overview but lack technical implementation details

**Step 2: Identify where reward signal information typically appears**
- Reward signals are a core component o

---
## Homework Challenges

**Challenge 1: Cross-Encoder Reranking** - Replace the LLM reranker with `cross-encoder/ms-marco-MiniLM-L-6-v2`. Compare speed and quality.

**Challenge 2: Multi-Query Retrieval** - Generate 3 variations of the question, search with each, union results. Compare with HyDE.

**Challenge 3: PageIndex on Your Own Document** - Upload your own PDF. Build the tree. How does it compare to the actual structure?

**Challenge 4: Vector RAG vs PageIndex Shootout** - Create 10 questions about the sample PDF. Run both approaches. Score manually.

**Challenge 5: Context Engineering** - Experiment with chunk sizes, number of retrieved chunks, ordering. How sensitive are results?

---

**References:**
- Context Rot: https://research.trychroma.com/context-rot
- ChatGPT Memory: https://manthanguptaa.in/posts/chatgpt_memory/
- PageIndex: https://pageindex.ai/blog/pageindex-intro | https://github.com/VectifyAI/PageIndex

**Next class: RLM (Recursive Language Models) - Context as environment. The model writes code to explore. The frontier.**